# Model Generation for Allen Visual dataset

This notebook generates 12 models as used in the analysis and in the report. It is meant to be run on colab for GPU usage.
1. Load Data
2. Train Models:
  - Single: 1 untrained + 5 Trained (mouse 4)
  - Multi: 1 untrained + 5 Trained (all mice)
3. Saves the models in Drive

Notes: The untraind models had _UT at the end while the trained did not have anything yet. The function lens.model.model_loader() works with this because that's how the models were trained during the semester. Future models should be trained using _TR at the end to simplify the task and thus the function should be modified to take this into account. Another viable option could be to distinguish UT from trained using the number of steps: number of steps = 1 -> UT, else TR.

In [ ]:
import os
import sys
sys.path.append('D:/EPFL/MA2/project/CEBRA-lens')
from cebra import CEBRA
import cebra
from src.cebra_lens import cebra_lens as lens
__DATADIR = "D:/EPFL/MA2/project/data"
os.environ["DATA_PATH"] = "D:/EPFL/MA2/project/models"

In [1]:
import cebra

In [10]:
import cebra

In [16]:
cebra.datasets.set_datapath("D:/EPFL/MA2/project/models")

In [17]:
cebra.datasets.get_data_root()

'data'

In [7]:
cebra.datasets.get_datapath()

'data'

In [3]:
#Set datapath where the data is stored
#because the data_root which is being used takes in the variable which is only once defined.
#why doesn't this work???
cebra.datasets.set_datapath(__DATADIR)


In [4]:
cebra.datasets.get_datapath()

'data'

# Load Data

Access single session data using train_datas[i].neural and .index for the DINO features.

This code uses Celia's function get_single_session_datasets().

In [5]:
cebra.datasets.get_datapath()

'data'

In [6]:
train_datas, valid_datas, discrete_labels_train, discrete_labels_val = (
    lens.utils_allen.get_single_session_datasets()
)

parameters = {
    "lr": 3e-4,
    "model_architechture": "offset10-model",
    "num_units": 32,
    "output_dim": 128,
    "temp": 1,
    "time_offsets": 10,
    "batch_size": 2042,
}


mice = ["mouse1", "mouse2", "mouse3", "mouse4"]

## Single session

In [7]:
import torch

In [8]:
torch.cuda.is_available()

True

In [10]:
max_iterations = 10000

In [13]:
# RUN MOUSE4 5 TIMES:
for i in range(5):

    cebra_model_single_session = CEBRA(
        model_architecture=parameters["model_architechture"],
        batch_size=parameters["batch_size"],
        learning_rate=parameters["lr"],
        temperature=parameters["temp"],
        num_hidden_units=parameters["num_units"],
        output_dimension=parameters["output_dim"],
        max_iterations=max_iterations,
        distance="cosine",
        conditional="time_delta",
        device="cuda:0",
        verbose=True,
        time_offsets=parameters["time_offsets"],
    )

    cebra_model_single_session.fit(train_datas[3].neural, train_datas[3].index)
    embeddings_single_session = cebra_model_single_session.transform(
        train_datas[3].neural
    )

    ################### SAVING ###################

    # torch version
    model_path = os.path.join(
        os.environ["DATA_PATH"],
        f"CEBRA_models/FinalModels/VISION/allen_single_session_{mice[3]}_{int(max_iterations/1000)}k_{i}_torch.pt",
    )
    cebra_model_single_session.save(model_path, backend="torch")
    print("Torch model saved under: ", model_path)
    # sklearn version
    model_path = os.path.join(
        os.environ["DATA_PATH"],
        f"CEBRA_models/FinalModels/VISION/allen_single_session_{mice[3]}_{int(max_iterations/1000)}k_{i}.pt",
    )
    cebra_model_single_session.save(model_path)
    print("Sklearn Model saved under: ", model_path)

pos: -0.9817 neg:  7.6586 total:  6.6769 temperature:  1.0000: 100%|██████████| 10000/10000 [07:12<00:00, 23.14it/s]


Torch model saved under:  D:/EPFL/MA2/project/models\CEBRA_models/FinalModels/VISION/allen_single_session_mouse4_10k_0_torch.pt
Sklearn Model saved under:  D:/EPFL/MA2/project/models\CEBRA_models/FinalModels/VISION/allen_single_session_mouse4_10k_0.pt


pos: -0.9823 neg:  7.6569 total:  6.6745 temperature:  1.0000:  85%|████████▍ | 8464/10000 [06:23<01:09, 22.05it/s]


KeyboardInterrupt: 

In [ ]:
# UNTRAINED
max_iterations = 1

cebra_model_single_session = CEBRA(
    model_architecture=parameters["model_architechture"],
    batch_size=parameters["batch_size"],
    learning_rate=parameters["lr"],
    temperature=parameters["temp"],
    num_hidden_units=parameters["num_units"],
    output_dimension=parameters["output_dim"],
    max_iterations=max_iterations,
    distance="cosine",
    conditional="time_delta",
    device="cuda_if_available",
    verbose=True,
    time_offsets=parameters["time_offsets"],
)

cebra_model_single_session.fit(train_datas[3].neural, train_datas[3].index)
embeddings_single_session = cebra_model_single_session.transform(train_datas[3].neural)

################### SAVING ###################

# torch version
model_path = os.path.join(
    os.environ["DATA_PATH"],
    f"CEBRA_models/FinalModels/VISION/allen_single_session_{mice[3]}_{int(max_iterations/1000)}k_UT_torch.pt",
)
cebra_model_single_session.save(model_path, backend="torch")
print("Torch model saved under: ", model_path)
# sklearn version
model_path = os.path.join(
    os.environ["DATA_PATH"],
    f"CEBRA_models/FinalModels/VISION/allen_single_session_{mice[3]}_{int(max_iterations/1000)}k_UT.pt",
)
cebra_model_single_session.save(model_path)
print("Sklearn Model saved under: ", model_path)

## Multi Session

In [ ]:
untrained = False

multi_session_neural_train = []
multi_session_index_train = []
multi_session_neural_valid = []
multi_session_index_valid = []

for i in range(4):
    multi_session_neural_train.append(train_datas[i].neural)
    multi_session_index_train.append(train_datas[i].index)

    multi_session_neural_valid.append(valid_datas[i].neural)
    multi_session_index_valid.append(valid_datas[i].index)


max_iterations = 10000

In [ ]:
# TRAIN 5 MULTI-SESSIONS

for i in range(5):
    # Multisession training
    cebra_model_multi_session = CEBRA(
        model_architecture=parameters["model_architechture"],
        batch_size=parameters["batch_size"],
        learning_rate=parameters["lr"],
        temperature=parameters["temp"],
        num_hidden_units=parameters["num_units"],
        output_dimension=parameters["output_dim"],
        max_iterations=max_iterations,
        distance="cosine",
        conditional="time_delta",
        device="cuda_if_available",
        verbose=True,
        time_offsets=parameters["time_offsets"],
    )

    # Provide a list of data, i.e. datas = [data_a, data_b, ...]
    cebra_model_multi_session.fit(multi_session_neural_train, multi_session_index_train)

    ################### SAVING ###################

    # torch version
    model_path = os.path.join(
        os.environ["DATA_PATH"],
        f"CEBRA_models/FinalModels/VISION/allen_multi_session_{int(max_iterations/1000)}k_{i}_torch.pt",
    )
    cebra_model_multi_session.save(model_path, backend="torch")
    print("Torch Model saved under: ", model_path)
    # sklearn version
    model_path = os.path.join(
        os.environ["DATA_PATH"],
        f"CEBRA_models/FinalModels/VISION/allen_multi_session_{int(max_iterations/1000)}k_{i}.pt",
    )
    cebra_model_multi_session.save(model_path)
    print("Sklearn Model saved under: ", model_path)

pos: -0.9804 neg:  9.0661 total:  8.0857 temperature:  1.0000: 100%|██████████| 10000/10000 [11:00<00:00, 15.14it/s]


Torch Model saved under:  /content/drive/MyDrive/CEBRA/Allen/CEBRA_models/celia_param/allen_multi_session_10k_3_torch.pt
Sklearn Model saved under:  /content/drive/MyDrive/CEBRA/Allen/CEBRA_models/celia_param/allen_multi_session_10k_3.pt


In [ ]:
# UNTRAINED

max_iterations = 1

# Multisession training
cebra_model_multi_session = CEBRA(
    model_architecture=parameters["model_architechture"],
    batch_size=parameters["batch_size"],
    learning_rate=parameters["lr"],
    temperature=parameters["temp"],
    num_hidden_units=parameters["num_units"],
    output_dimension=parameters["output_dim"],
    max_iterations=max_iterations,
    distance="cosine",
    conditional="time_delta",
    device="cuda_if_available",
    verbose=True,
    time_offsets=parameters["time_offsets"],
)

# Provide a list of data, i.e. datas = [data_a, data_b, ...]
cebra_model_multi_session.fit(multi_session_neural_train, multi_session_index_train)

################### SAVING ###################

# torch version
model_path = os.path.join(
    os.environ["DATA_PATH"],
    f"CEBRA_models/FinalModels/VISION/allen_multi_session_{int(max_iterations/1000)}k_UT_torch.pt",
)
cebra_model_multi_session.save(model_path, backend="torch")
print("Torch Model saved under: ", model_path)
# sklearn version
model_path = os.path.join(
    os.environ["DATA_PATH"],
    f"CEBRA_models/FinalModels/VISION/allen_multi_session_{int(max_iterations/1000)}k_UT.pt",
)
cebra_model_multi_session.save(model_path)
print("Sklearn Model saved under: ", model_path)